# Lab 4 - logistic regression, curves and a chosen threshold

**Session 4.** Fit a classifier with scikit-learn, read the ROC and PR curves, then pick a
threshold from a stated cost ratio and defend it. The defence is the deliverable.

## 1. A credit-screening problem

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1200, n_features=8, n_informative=4, n_redundant=1,
    weights=[0.88, 0.12],          # 12% positives - imbalanced, as these problems are
    class_sep=0.9, random_state=2026,
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=2026)   # stratify: keep the base rate
print(f"train {X_tr.shape}, test {X_te.shape}, base rate {y.mean():.3f}")

## 2. Fit, in a pipeline

The scaler belongs inside the pipeline even now - session 9 explains why in detail.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([("sc", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=1000))])
model.fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]
print("first five predicted probabilities:", p_te[:5].round(3))

## 3. Accuracy is the wrong headline

Compare the model against the majority-class baseline before believing anything.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print(f"majority-class baseline accuracy : {1 - y_te.mean():.3f}")
print(f"model accuracy at threshold 0.5  : {accuracy_score(y_te, p_te >= 0.5):.3f}")
print()
print(classification_report(y_te, p_te >= 0.5, digits=3))

## 4. ROC and precision-recall

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score, roc_curve)

fpr, tpr, _ = roc_curve(y_te, p_te)
prec, rec, _ = precision_recall_curve(y_te, p_te)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="false positive rate", ylabel="recall",
            title=f"ROC (AUC = {roc_auc_score(y_te, p_te):.3f})")
axes[1].plot(rec, prec)
axes[1].axhline(y_te.mean(), ls="--", color="grey")   # the no-skill line IS the base rate
axes[1].set(xlabel="recall", ylabel="precision",
            title=f"PR (AP = {average_precision_score(y_te, p_te):.3f})")
plt.tight_layout()
plt.show()

## 5. The threshold is a cost decision

Suppose a missed default (false negative) costs 10 times a needless review (false
positive). Then the expected cost of a threshold is `10 * FN + 1 * FP`, and the best
threshold is simply the one that minimises it.

In [ ]:
from sklearn.metrics import confusion_matrix

COST_FN, COST_FP = 10, 1
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    tn, fp, fn, tp = confusion_matrix(y_te, p_te >= t).ravel()
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    rows.append((t, tp, fp, fn, precision, recall, COST_FN * fn + COST_FP * fp))

print(" thresh   TP   FP   FN   precision  recall    cost")
for t, tp, fp, fn, precision, recall, cost in rows:
    print(f"  {t:.2f}  {tp:4d} {fp:4d} {fn:4d}    {precision:6.3f}  {recall:6.3f}  {cost:6d}")

best = min(rows, key=lambda r: r[-1])
print(f"\ncost-minimising threshold: {best[0]:.2f} (cost {best[-1]}), "
      f"versus cost {[r for r in rows if abs(r[0] - 0.5) < 1e-9][0][-1]} at the 0.5 default")

## Exercises

1. **Flip the costs.** Set `COST_FN = 1, COST_FP = 10` and re-run. Where does the threshold
   move, and does the *model* change at all?
2. **Calibration.** Plot `sklearn.calibration.CalibrationDisplay` for this model. Are the
   probabilities usable as probabilities, or only as a ranking?
3. **Write the defence.** Two sentences naming the cost ratio you assumed, the threshold it
   implies, and the resulting recall. This is exactly what Assignment 2 asks for, so get the
   feedback here.